# A1/A2 Data Inventory & EDA (CPU)

Menjalankan *source-file inventory* dan *exploratory data analysis* secara
reproduktif lewat logika `sipature_ml` — tanpa duplikasi bisnis-logic di notebook.
Ikuti `docs/eda-report.md`, `docs/data-inventory.md`, dan
`docs/reproducibility-runbook.md` sebelum eksekusi.

Notebook ini tidak membaca split/label apa pun; murni profiling data mentah.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DATASET_SOURCE_DIR = DRIVE_ROOT / "data" / "raw"  # raw CSV sumber di Drive

PROJECT_DIR = Path("/content/hackathon/ml")
LOCAL_DATASET_DIR = PROJECT_DIR / "data" / "raw"

REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "eda"

DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "eda"

SOURCE_ENCODING = "utf-8-sig"

print("Drive root:", DRIVE_ROOT)
print("Sumber dataset:", DATASET_SOURCE_DIR)
print("Dataset lokal:", LOCAL_DATASET_DIR)


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

# GitHub menerima format Basic: x-access-token:<token>
credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


In [ ]:
import numpy
import pandas
import pyarrow
import sklearn
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)


In [ ]:
# Salin raw CSV sumber dari Drive ke lokal agar sipature_ml membaca
# dari path yang deterministik.
import shutil
from pathlib import Path

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_SOURCE_DIR.is_dir(), (
    f"Sumber dataset tidak ditemukan di Drive: {DATASET_SOURCE_DIR}\n"
    "Unggah CSV mentah ke folder tersebut sebelum melanjutkan."
)

copied = []
for source in sorted(DATASET_SOURCE_DIR.glob("*.csv")):
    destination = LOCAL_DATASET_DIR / source.name
    shutil.copy2(source, destination)
    copied.append(source.name)
    print("Disalin:", source.name)

print("\nTotal file:", len(copied))


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


In [ ]:
from sipature_ml.config import load_config
from sipature_ml.environment import build_environment_snapshot

config = load_config("pipeline")
snapshot = build_environment_snapshot()

print("Pipeline version:", config["pipeline_version"])
print("Seed:", config["seed"])
print("Git commit:", snapshot["git_commit"])
print("Git dirty:", snapshot["git_dirty"])
print("Python:", snapshot["python"])


In [ ]:
from sipature_ml.inventory import inventory_dataset, write_inventory

inventory = inventory_dataset(LOCAL_DATASET_DIR, encoding=SOURCE_ENCODING)

print("Dataset dir:", inventory["dataset_dir"])
print("File count:", inventory["file_count"])

json_path, csv_path = write_inventory(inventory, REPORT_DIR)
print("Inventory JSON:", json_path)
print("Inventory CSV :", csv_path)


In [ ]:
# Tampilkan ringkasan inventory agar masalah terbaca langsung di notebook.
import pandas as pd

inventory_df = pd.DataFrame(inventory["files"])
inventory_df


In [ ]:
from sipature_ml.eda import run_eda

eda_summary = run_eda(LOCAL_DATASET_DIR, REPORT_DIR, FIGURE_DIR)

print("Figures:", len(eda_summary["figures"]))
for name in eda_summary["figures"]:
    print("-", name)
print("\nReview summary keys:", sorted(eda_summary["review_summary"]))


In [ ]:
# Salin output inventory + EDA ke Drive agar menjadi artefak persisten.
import shutil
from pathlib import Path

DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

report_files = [
    "data_inventory.json",
    "data_inventory.csv",
    "eda_summary.json",
    "eda_file_profile.csv",
    "eda_candidate_aspects.csv",
    "eda_place_coverage.csv",
    "eda_metadata_completeness.csv",
    "eda_coordinates.csv",
    "eda_ngrams.csv",
    "eda_service_density.csv",
]

for filename in report_files:
    source = REPORT_DIR / filename
    if source.is_file():
        shutil.copy2(source, DRIVE_REPORT_DIR / filename)
        print("Report disalin:", filename)

for name in eda_summary["figures"]:
    source = FIGURE_DIR / name
    if source.is_file():
        shutil.copy2(source, DRIVE_FIGURE_DIR / name)
        print("Figure disalin:", name)


In [ ]:
# ============================================================
# RUN SUMMARY — output path, hash sumber, dan limitations.
# ============================================================
import json
from sipature_ml.manifest import sha256_file

print("SOURCE HASHES:")
for name, digest in sorted(eda_summary["source_files"].items()):
    print(f"  {digest}  {name}")

print("\nOUTPUT REPORT DIR:", REPORT_DIR)
print("OUTPUT FIGURE DIR :", FIGURE_DIR)
print("DRIVE REPORT DIR  :", DRIVE_REPORT_DIR)
print("DRIVE FIGURE DIR  :", DRIVE_FIGURE_DIR)

print("\nEDA VERSION:", eda_summary["eda_version"])
print("LIMITATIONS:")
for item in eda_summary["limitations"]:
    print("  -", item)
